[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/rattleback/rattleback.ipynb)

# The Rattleback

This example simulates the rolling motion of a rattleback, a boat-shaped top that converts rocking into spin and reverses spin of the unfavourable sense. The body is modelled as a rigid hull whose ellipsoidal underside rolls without slipping on a horizontal table while inset steel balls skew its principal axes of inertia away from the curvature axes of the hull. That misalignment couples pitching and rolling to vertical spin, causing rotation in one sense to grow at the expense of rocking while rotation in the opposite sense destabilizes into rocking and turns around.

Translating the body spin plane from the centre of mass to the instantaneous contact point builds the rolling map `rolling = contact_shift >> Spin`, which sends each spin to the rigid rate holding the contact point at rest. Pulling loads back through `rolling.adjugate()` eliminates the unknown table reaction because forces through a resting contact do no work on rolling rates, while `rolling.adjugate()(inertia(rolling))` combines translational and rotational inertia into one reduced map on spins. Differentiating the ellipsoid support point as the hull rocks supplies the velocity-dependent rate shift `rolling_change` alongside `inertia(rate).commutator(rate)`.

In the matrix Newton-Euler notation of [Arioli and Koch (2022)](https://link.springer.com/article/10.1007/s00332-022-09797-7) and [Yoshida, Tokieda and Morrison (2017)](https://arxiv.org/abs/1609.09223), the split translational and rotational balances $m(\dot{\mathbf{v}} + \boldsymbol{\omega}\times\mathbf{v}) = \mathbf{f}_c + m R^\top\mathbf{g}$ and $I\dot{\boldsymbol{\omega}} + \boldsymbol{\omega}\times(I\boldsymbol{\omega}) = \mathbf{r}_c\times\mathbf{f}_c$ under the no-slip constraint $\mathbf{v} = \mathbf{r}_c\times\boldsymbol{\omega}$ at the contact lever $\mathbf{r}_c(\mathbf{n}) = \mathbf{c} - A^2\mathbf{n}/\sqrt{\mathbf{n}^\top A^2\mathbf{n}}$ eliminate the contact reaction $\mathbf{f}_c$ through the parallel-axis matrix $I_c(\mathbf{n}) = I + m\bigl((\mathbf{r}_c^\top\mathbf{r}_c)I_3 - \mathbf{r}_c\mathbf{r}_c^\top\bigr)$ to give $I_c(\mathbf{n})\dot{\boldsymbol{\omega}} = (I\boldsymbol{\omega})\times\boldsymbol{\omega} + m\,\mathbf{r}_c\times(\dot{\mathbf{r}}_c\times\boldsymbol{\omega}) - m\,\mathbf{r}_c\times\bigl((\mathbf{r}_c\times\boldsymbol{\omega})\times\boldsymbol{\omega}\bigr) + m g\,\mathbf{r}_c\times\mathbf{n}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_states
from examples.mechanics.rattleback import render

ctx = NumpyContext(ga)
mv = ctx.multivector

Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
Point = ga.gatype.antivector()
Direction = Point.derive.degenerate
Spin = ga.gatype.bivector().derive.nondegenerate
Rate = ga.gatype.bivector()
Motor = ga.gatype.rotor()
Forque = ga.gatype.antibivector()
Shape = ga.gatype((Direction, Plane))
Rolling = ga.gatype((Rate, Spin))
Inertia = ga.gatype((Forque, Rate))
Quadric = ga.gatype((Plane, Point))

infinity = mv.w                                                       # [] Plane+
origin = infinity.dual()                                              # [] Point-
up = mv.z.dual()                                                      # [] Direction-
mirrors = stack((mv.x, mv.y, mv.z))                                   # [3] Plane+

## Body geometry and inertia

A wooden hull with an ellipsoidal underside, a flat deck, two corner chamfers, and two inset steel balls placed half a turn apart around the vertical breaks reflection symmetry across both horizontal axes of the shell. Each solid region is represented by quadrics whose pairing `quadric(point) & point` is negative inside and zero on the boundary, where `quadric(point)` gives the outward tangent plane. Sampling a regular grid against those quadrics locates the centre of mass and sums each interior point's regressive inertia `(body_points * sample_masses).regressive(body_points.commutator(Rate))`, while the GPU ray tracer renders the exact same quadrics.

In [ ]:
semi_axes = np.array([5, 17 / 16, 1])
mass = 4
gravity_strength = 9.8
deck_height = 0.5
chamfer_roll = np.deg2rad(35)
chamfer_pitch = np.deg2rad(30)
chamfer_point = [3.2, 0, 0.2]
ball_radius = 0.4
ball_fore_offset = 3
ball_sink = 0.6
wood_density = 0.7
steel_density = 7.85
sample_spacing = 0.15

In [ ]:
def half_space(plane: Plane) -> Quadric:
    """Symmetrizing `plane` with `infinity` forms a quadric negative on `plane`'s inner side (`[...] Plane+ <- Point-`)."""
    return (plane * (infinity & Point) + infinity * (plane & Point)) / 2


def ball(centre: Point, radius: float) -> Quadric:
    """Coordinate planes offset through unit-weight `centre` points define solid sphere quadrics (`[...] Plane+ <- Point-`) of `radius`."""
    sphere_planes = mirrors - infinity * (mirrors & centre[..., None])  # [..., 3] Plane+
    return (sphere_planes * (sphere_planes & Point)).sum(axis=-1) / radius**2 - infinity * (infinity & Point)


# Stretch the unit sphere onto the ellipsoidal hull and pull its axial planes back through the adjugate.
directions = mirrors.dual()                                           # [3] Direction-
outline = (directions * (mirrors & Direction) * semi_axes).sum(axis=0)  # [] Direction- <- Direction-
shape = (directions * (Plane & directions) * semi_axes**2).sum(axis=0)  # [] Direction- <- Plane+
shell_planes = outline.adjugate().solve(mirrors)                      # [3] Plane+
underside = (shell_planes * (shell_planes & Point)).sum(axis=0) - infinity * (infinity & Point)  # [] Plane+ <- Point-
deck = half_space(mv.z - infinity * deck_height)                      # [] Plane+ <- Point-

# Place the two corner chamfers and steel balls half a turn apart around the vertical axis.
side_turns = (mv.xy * np.array([0, np.pi]) / 2).exp()                 # [2] Motor+
chamfer_turn = (mv.yz * (-chamfer_roll / 2)).exp() * (mv.xz * (chamfer_pitch / 2)).exp()  # [] Motor+
chamfer_normal = chamfer_turn >> mv.z                                 # [] Plane+
chamfer_anchor = origin + mv.direction(chamfer_point)                 # [] Point-
chamfer_planes = side_turns >> (chamfer_normal - infinity * (chamfer_normal & chamfer_anchor))  # [2] Plane+
chamfers = half_space(chamfer_planes)                                 # [2] Plane+ <- Point-
hull_half_width = semi_axes[1] * np.sqrt(1 - (ball_fore_offset / semi_axes[0])**2)
ball_seat = origin + mv.direction([ball_fore_offset, hull_half_width + ball_radius * (1 - 2 * ball_sink), 0])  # [] Point-
balls = ball(side_turns >> ball_seat, ball_radius)                    # [2] Plane+ <- Point-
wood = concatenate([underside[None], deck[None], chamfers, -balls])   # [6] Plane+ <- Point-

In [ ]:
def mass_distribution(wood: Quadric, balls: Quadric, grid: Point, wood_density: float,
                      steel_density: float, mass: float) -> tuple[Point, Point, Inertia]:
    """Interior grid points (`[...] Point-`) weighted by wood and steel densities determine the centre of mass (`[] Point-`) and body inertia (`[] Forque- <- Rate+`)."""
    wood_mask = ((wood[:, None](grid) & grid) <= 0).all(axis=0)
    steel_mask = ((balls[:, None](grid) & grid) <= 0).any(axis=0)
    densities = wood_density * wood_mask + steel_density * steel_mask
    body_mask = densities > 0
    samples = grid[body_mask]                                         # [...] Point-
    sample_masses = mass * densities[body_mask] / densities[body_mask].sum()
    mass_centre = (samples * (sample_masses / mass)).sum(axis=0)      # [] Point-
    # Joining each mass-weighted point with its velocity under Rate sums the translational and rotational inertia.
    body_points = origin + (samples - mass_centre)                    # [...] Point-
    inertia = (body_points * sample_masses).regressive(body_points.commutator(Rate)).sum(axis=0)  # [] Forque- <- Rate+
    return samples, mass_centre, inertia


low = -semi_axes - [0, 2 * ball_radius, 0]
high = np.array([semi_axes[0], semi_axes[1] + 2 * ball_radius, deck_height])
grid_axes = [np.arange(start, stop, sample_spacing) + sample_spacing / 2 for start, stop in zip(low, high)]
grid_coords = np.stack(np.meshgrid(*grid_axes, indexing="ij"), axis=-1).reshape(-1, 3)
n_grid = len(grid_coords)
grid = mv.point(grid_coords)                                          # [n_grid] Point-
samples, mass_centre, inertia = mass_distribution(wood, balls, grid, wood_density, steel_density, mass)
n_samples = samples.shape[0]

# Shift the quadrics and sample points so the centre of mass sits at the origin.
centre_shift = ((origin - mass_centre) * origin.inverse() / 2).exp()  # [] Motor+
wood = centre_shift >> wood(centre_shift << Point)                    # [6] Plane+ <- Point-
balls = centre_shift >> balls(centre_shift << Point)                  # [2] Plane+ <- Point-
samples = centre_shift >> samples                                     # [n_samples] Point-
centre = origin - mass_centre                                         # [] Direction-
gravity = -up * gravity_strength                                      # [] Direction-

table_plane = up.dual_inverse()                                       # [] Plane+
rest_height = (table_plane & shape(table_plane)).square_root() - (table_plane & centre)  # [] Scalar+
rest_placement = (up * origin.inverse() * (rest_height / 2)).exp()    # [] Motor+
save_figure(render.setup_figure(wood, balls, samples, rest_placement), "rattleback_body");

The painted teal underside, bare-wood deck, angled corner chamfers, and mirrored steel balls on opposite flanks skew the principal axes of inertia away from the symmetry planes of the ellipsoidal hull.

## Rolling inertia and rocking into spin

Because the material point touching the table is instantaneously at rest, power invariance `rolling.adjugate()(force) & spin == force & rolling(spin)` annihilates the table reaction and pulls `inertia` back to `rolling.adjugate()(inertia(rolling))` on body spins. As the hull rocks, a new point on the ellipsoid continually takes over contact, contributing `rolling_change` to the rate derivative even at constant spin. Released from rest at a fifteen-degree pitch tilt, the hull rocks and rolls until its raised potential energy transfers into steady counterclockwise spin. Simulating and ray-tracing this animation takes about ten seconds.

In [ ]:
rocking_tilt = np.deg2rad(15)
dt = 0.04
n_rocking_steps = 200
duration_ms = round(dt * 1000)

In [ ]:
def contact_geometry(motor: Motor, shape: Shape, centre: Direction,
                     up: Direction) -> tuple[Plane, Direction, Scalar, Rolling]:
    """The body-frame table normal determines the ellipsoid support offset, hull centre height, and rolling map (`[...] Rate+ <- Spin+`)."""
    normal = (motor << up).dual_inverse()                             # [...] Plane+
    height = (normal & shape(normal)).square_root()                   # [...] Scalar+
    support = -shape(normal) / height                                 # [...] Direction-

    # Translating a spin from the centre of mass to the contact point turns the body instantaneously about the contact.
    contact_shift = ((centre + support) * origin.inverse() / 2).exp()  # [...] Motor+
    rolling = contact_shift >> Spin                                   # [...] Rate+ <- Spin+
    return normal, support, height, rolling


def dynamics(motor: Motor, spin: Spin, shape: Shape, centre: Direction, inertia: Inertia,
             mass: float, gravity: Direction, up: Direction) -> tuple[Rate, Spin]:
    """Pulling the force and Euler balance back across the rolling map gives the body rate (`[...] Rate+`) and angular acceleration (`[...] Spin+`)."""
    normal, support, height, rolling = contact_geometry(motor, shape, centre, up)
    rate = rolling(spin)                                              # [...] Rate+

    # Differentiate the support point as the table normal rotates in the body while the hull centre stays fixed.
    normal_rate = spin.commutator(normal)                             # [...] Plane+
    height_rate = (normal & shape(normal_rate)) / height              # [...] Scalar+
    contact_rate = -shape(normal_rate) / height - support * height_rate / height  # [...] Direction-
    rolling_change = (contact_rate * origin.inverse()).commutator(spin)  # [...] Rate+
    force = (origin & (motor << gravity)) * mass                      # [...] Forque-

    # Pulling forques back across the rolling map annihilates the workless reaction at the resting contact.
    pullback = rolling.adjugate()                                     # [...] Forque- <- Forque-
    effective_inertia = pullback(inertia(rolling))                    # [...] Forque- <- Spin+
    acceleration = effective_inertia.solve(pullback(
        force - inertia(rate).commutator(rate) - inertia(rolling_change)
    ))                                                                # [...] Spin+
    return rate, acceleration

In [ ]:
def rkmk_step(motor: Motor, spin: Spin, dt: float, shape: Shape, centre: Direction,
              inertia: Inertia, mass: float, gravity: Direction, up: Direction) -> tuple[Motor, Spin]:
    """A fourth-order Runge-Kutta-Munthe-Kaas step advances `motor` and body-frame `spin` over `dt`."""
    stage_increment = (spin * 0).retyped(Rate)                        # [...] Rate+
    stage_spin = spin                                                 # [...] Spin+
    total_increment = stage_increment                                 # [...] Rate+
    total_spin_change = spin * 0                                      # [...] Spin+
    for step_fraction, weight in ((0.5, 1 / 6), (0.5, 1 / 3), (1.0, 1 / 3), (0.0, 1 / 6)):
        rate, acceleration = dynamics(motor * stage_increment.exp(), stage_spin,
                                      shape, centre, inertia, mass, gravity, up)
        adjoint = stage_increment.commutator(Rate) * 2                # [...] Rate+ <- Rate+
        dexp_inverse = Rate + adjoint / 2 + adjoint(adjoint) / 12     # [...] Rate+ <- Rate+
        increment_rate = dexp_inverse(-rate / 2)                      # [...] Rate+
        total_increment = total_increment + increment_rate * (dt * weight)  # [...] Rate+
        total_spin_change = total_spin_change + acceleration * (dt * weight)  # [...] Spin+
        stage_increment = increment_rate * (dt * step_fraction)       # [...] Rate+
        stage_spin = spin + acceleration * (dt * step_fraction)       # [...] Spin+
    return motor * total_increment.exp(), spin + total_spin_change


def trajectory(motor: Motor, spin: Spin, dt: float, shape: Shape, centre: Direction,
               inertia: Inertia, mass: float, gravity: Direction,
               up: Direction) -> Iterator[tuple[Motor, Spin]]:
    """Yields rigid placements (`[...] Motor+`) and body-frame spins (`[...] Spin+`) at steps of `dt`."""
    while True:
        yield motor, spin
        motor, spin = rkmk_step(motor, spin, dt, shape, centre, inertia, mass, gravity, up)


rocking_spin = mv.xy * 0                                              # [] Spin+
rocking_tilt_motor = (-mv.xz * (rocking_tilt / 2)).exp()              # [] Motor+
rocking_normal = (rocking_tilt_motor << up).dual_inverse()            # [] Plane+
rocking_height = (rocking_normal & shape(rocking_normal)).square_root() - (rocking_normal & centre)  # [] Scalar+
rocking_motor = (up * origin.inverse() * (rocking_height / 2)).exp() * rocking_tilt_motor  # [] Motor+

rocking_states = show_states(
    islice(trajectory(rocking_motor, rocking_spin, dt, shape, centre, inertia, mass, gravity, up), n_rocking_steps + 1),
    render.frames, wood, balls, samples,
    name="rattleback_rocking", duration_ms=duration_ms, n_frames=n_rocking_steps + 1,
);

Pitching oscillations excite rolling through the misaligned inertia and drain steadily into counterclockwise yaw as the flat deck levels out.

## Spin reversal

Launched spinning clockwise with a slight pitching disturbance, the hull pumps its kinetic energy into large rocking oscillations, comes momentarily to rest in yaw, and spins up in the opposite counterclockwise direction. Simulating and ray-tracing this animation takes about twelve seconds.

In [ ]:
reversal_spin_rate = -1.5
reversal_pitch_rate = 0.1
n_reversal_steps = 250

In [ ]:
def energy(motor: Motor, spin: Spin, shape: Shape, centre: Direction, inertia: Inertia,
           mass: float, gravity: Direction, up: Direction) -> Scalar:
    """Kinetic and gravitational potential contributions sum to the total mechanical energy (`[...] Scalar+`) of the rolling body."""
    _, _, _, rolling = contact_geometry(motor, shape, centre, up)
    rate = rolling(spin)                                              # [...] Rate+
    potential = -mass * (gravity.dual_inverse() & (motor >> origin))  # [...] Scalar+
    return (rate & inertia(rate)) / 2 + potential


reversal_spin = mv.xy * reversal_spin_rate - mv.xz * reversal_pitch_rate  # [] Spin+
reversal_normal = up.dual_inverse()                                   # [] Plane+
reversal_height = (reversal_normal & shape(reversal_normal)).square_root() - (reversal_normal & centre)  # [] Scalar+
reversal_motor = (up * origin.inverse() * (reversal_height / 2)).exp()  # [] Motor+

reversal_states = show_states(
    islice(trajectory(reversal_motor, reversal_spin, dt, shape, centre, inertia, mass, gravity, up), n_reversal_steps + 1),
    render.frames, wood, balls, samples,
    name="rattleback_reversal", duration_ms=duration_ms, n_frames=n_reversal_steps + 1,
);

The clockwise spin amplifies the initial wobble until the hull rocks almost end to end, halting the initial rotation before converting the rocking energy back into sustained counterclockwise spin.

In [ ]:
# checks
rocking_motors, rocking_spins = (stack(items) for items in zip(*rocking_states))
reversal_motors, reversal_spins = (stack(items) for items in zip(*reversal_states))
rocking_vertical_spin = -(rocking_motors >> rocking_spins).scalar_product(mv.xy)    # [n_rocking_steps + 1] Scalar+
reversal_vertical_spin = -(reversal_motors >> reversal_spins).scalar_product(mv.xy)  # [n_reversal_steps + 1] Scalar+
rocking_energies = energy(rocking_motors, rocking_spins, shape, centre, inertia, mass, gravity, up)      # [n_rocking_steps + 1] Scalar+
reversal_energies = energy(reversal_motors, reversal_spins, shape, centre, inertia, mass, gravity, up)   # [n_reversal_steps + 1] Scalar+

assert rocking_vertical_spin[0].abs() < 1e-11 and rocking_vertical_spin[-1] > 1.4
assert reversal_vertical_spin[0] < -1.4 and reversal_vertical_spin[-1] > 1.4
assert ((rocking_energies / rocking_energies[0] - 1).abs() < 1e-2).all()
assert ((reversal_energies / reversal_energies[0] - 1).abs() < 1e-2).all()